## Integrador API OPTA

In [2]:
import requests
import re
from bs4 import BeautifulSoup
import os
import json
from datetime import datetime
import time
import random
import pandas as pd
import re
from datetime import datetime, timedelta,date
import warnings  # Importa el módulo warnings, que maneja los mensajes de advertencia en Python.
import sw_scraping_fun as swf
import sw_json2csv_fun as swj


with open("config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

ruta_dest=config['dummy_ori']
liga_seleccionada= config['dummy_competition']

warnings.filterwarnings("ignore")



In [3]:
def extraer_datos_partido(match,date,home,away,token):
    df_game=swf.get_datos_partido(match,date,home,away,token)
    data=swj.get_events(df_game)

    return data

# Scraping + Parseo

In [5]:
metadata = pd.read_excel("config/metadata.xlsx")

## API -> JSON

Se extrae un fichero json de cada partido. Posteriormente, se parsea y se transforma en csv.
Un partido contiene la siguiente información, a nivel partido:
- *matchdata*: atributos del partido
- *playerdata*: atributos de jugadores
- *teamdata*: atributos de los equipos
- *eventdata*: eventing

Adicionalmente, se incorporan datos agregados de equipos (teamstats), extraidos mediante selenium.

In [7]:
ficheros = [f for f in os.listdir(os.path.join(os.getcwd(),"fixtures")) if f.endswith(('.xlsx', '.xls')) and "$" not in f]
fixtures = pd.DataFrame()
for i in ficheros:
    fi = pd.read_excel(os.path.join(os.getcwd(),"fixtures",i))
    fixtures=pd.concat([fixtures,fi])

In [8]:
for i,j in metadata[metadata.competicion_id==liga_seleccionada].iterrows():
        ruta_dest = j['origen']
        season= j['season']
        comp = j['competicion_id']
        #comps.append(comp)
        #seasons.append(season)
        fixtures_url = j['url']
        fichero_liga = "matches_{}_{}.xlsx".format(comp.lower(),season)
    
    
        
        if fichero_liga in ficheros:
            all_fixtures_df=pd.read_excel(os.path.join(os.getcwd(),"fixtures")+"/{}".format(fichero_liga))
            all_fixtures_df = all_fixtures_df.drop_duplicates(subset="match_id",keep='first')
        else:
            all_fixtures_df = swf.scrape_fixtures(metadata,fixtures_url)
            if all_fixtures_df.shape[0]>0:
                all_fixtures_df['season'] = season
                all_fixtures_df['competition'] = comp
        all_fixtures_df['date'] = all_fixtures_df['date'].fillna(all_fixtures_df[all_fixtures_df['date'].isna()==False]['date'].values[-1])
        all_fixtures_df['dia_id']=all_fixtures_df.date.apply(lambda x: datetime.fromisoformat(x.replace("Z", "")).date())

In [9]:
df_partidos = all_fixtures_df[all_fixtures_df.dia_id<date.today()]

print("Partidos ya jugados: {} ({} totales en {})".format(df_partidos.shape[0],all_fixtures_df.shape[0],liga_seleccionada))

Partidos ya jugados: 168 (240 totales en esp-ligaf)


In [10]:
df_partidos.head()

,competition,season,date,home_team,away_team,stadium,match_id,URL,dia_id
72,esp-ligaf,2025-2026,2026-02-22Z,Sevilla,Atlético Madrid,NaN,8bo8kwuy2uvdmay9yf6dq9ses,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
73,esp-ligaf,2025-2026,2026-02-22Z,Real Sociedad,Espanyol,NaN,8cuardpf2jimws64c0qsc0xlg,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
74,esp-ligaf,2025-2026,2026-02-22Z,Eibar,Athletic Club,NaN,8b8qojrl7pmleq86uh9y0bhn8,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
75,esp-ligaf,2025-2026,2026-02-22Z,Real Madrid,Tenerife,NaN,8agzaoxboug1td3uvjrg6obh0,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-22
76,esp-ligaf,2025-2026,2026-02-21Z,Granada,Barcelona,NaN,8d879ol25noswi79mfshf9rf8,https://www.scoresway.com/en_GB/soccer/primera...,2026-02-21


In [11]:
data=extraer_datos_partido(df_partidos.match_id.values[-1],all_fixtures_df.date.values[-1],
                     all_fixtures_df.home_team.values[-1],
                     all_fixtures_df.away_team.values[-1],
                    swf.obtener_sdapi_outlet_key(metadata.url.values[0]))

Error: Añadiendo token por defecto
Leyendo sdapi_outlet_key desde config/config.json...
https://api.performfeeds.com/soccerdata/matchevent/ft1tiv1inq7v1sk3y9tv12yh5/6jotswzwmrnex0mktyd575ez8?_rt=c&_lcl=en&_fmt=jsonp&sps=widgets&_clbk=W3e14cbc3e4b2577e854bf210e5a3c7028c7409678
⏳ Esperando 5.13 segundos antes de descargar (intento 1/3)...
✅ Guardado: 2025-08-30Z_Athletic Club_Tenerife_6jotswzwmrnex0mktyd575ez8.json


In [12]:
data.keys()

dict_keys(['eventData', 'matchData', 'teamData', 'playerData'])

#### Eventos

In [14]:
events= data['eventData']
print("Nº filas: {}".format(events.shape[0]))
events.head()

Nº filas: 1463


,id,eventId,typeId,periodId,timeMin,timeSec,contestantId,outcome,x,y,...,relatedEventId,relatedPlayerId,is_penalty,isGoal,minute,second,oppositionTeamName,time_seconds,position,isFirstEleven
0,2840786093,1,34,16,0,0,1azk57wev30h044w949j46kpu,1,0.0,0.0,...,NaN,NaN,0,0,0,0,Tenerife,0,NaN,NaN
1,2840786451,1,34,16,0,0,215mjv43m1cuncum9o4189mhg,1,0.0,0.0,...,NaN,NaN,0,0,0,0,Athletic Club,0,NaN,NaN
2,2840794715,2,32,1,0,0,215mjv43m1cuncum9o4189mhg,1,0.0,0.0,...,NaN,NaN,0,0,0,0,Athletic Club,0,NaN,NaN
3,2840794705,2,32,1,0,0,1azk57wev30h044w949j46kpu,1,0.0,0.0,...,NaN,NaN,0,0,0,0,Tenerife,0,NaN,NaN
4,2840853617,781,1,1,0,1,1azk57wev30h044w949j46kpu,1,49.9,49.9,...,NaN,NaN,0,0,0,1,Tenerife,1,FW,1.0


#### Dimensiones:

- Partido
- Equipos
- Jugadores

In [16]:
df_match= data['matchData']
df_match.head()

,matchId,id,coverageLevel,date,time,localDate,localTime,week,numberOfPeriods,periodLength,...,away_code,away_position,competition_id,competition_name,competition_knownName,competition_competitionCode,competition_competitionFormat,venueName,expandedMaxMinute,matchStatus
0,6jotswzwmrnex0mktyd575ez8,6jotswzwmrnex0mktyd575ez8,13,2025-08-30Z,10:00:00Z,2025-08-30,12:00:00,1,2,45,...,TEN,away,5k620c7y6dlbmcm88dt3eb7t,Primera División Femenina,Primera División Femenina,PDF,Domestic league,Instalaciones de Lezama Campo 2,96,Played


In [17]:
df_teams= data['teamData']
df_teams.head()

,teamId,teamName,shortName,officialName,code,field,country,matchId,scores.halftime,scores.fulltime
0,1azk57wev30h044w949j46kpu,Athletic Club,Athletic,Athletic Club,ATH,home,Spain,6jotswzwmrnex0mktyd575ez8,0,0
1,215mjv43m1cuncum9o4189mhg,Tenerife,Tenerife,CD Tenerife,TEN,away,Spain,6jotswzwmrnex0mktyd575ez8,0,0


In [18]:
df_players= data['playerData']
print("Nº filas: {}".format(df_players.shape[0]))
df_players.head()

Nº filas: 41


,field_position,playerId,value_TeamPlayerFormation,shirtNo,value_TeamFormation,teamId,matchId,field,isFirstEleven,position,playerName,subbedOutPlayerId,subbedInExpandedMinute,subbedInPeriod_value,subbedInPlayerId,subbedOutExpandedMinute,subbedOutPeriod_value
0,1,ea2eq21d08asfeqh17vcxpwmi,1,13,4,1azk57wev30h044w949j46kpu,6jotswzwmrnex0mktyd575ez8,home,1,GK,Adriana Nanclares,NaN,NaN,NaN,NaN,NaN,NaN
1,2,ef8jptd63fi9rll1e1gx61z6y,2,20,4,1azk57wev30h044w949j46kpu,6jotswzwmrnex0mktyd575ez8,home,1,DR,Ane Elexpuru,NaN,NaN,NaN,NaN,NaN,NaN
2,2,f0m3ncnvrrrj3wcgzrz7ja44q,3,17,4,1azk57wev30h044w949j46kpu,6jotswzwmrnex0mktyd575ez8,home,1,DC,Nerea Nevado,NaN,NaN,NaN,NaN,NaN,NaN
3,3,4by0xo4xinb0m8nziofw0g3bp,4,14,4,1azk57wev30h044w949j46kpu,6jotswzwmrnex0mktyd575ez8,home,1,MC,Leire Baños,NaN,NaN,NaN,NaN,NaN,NaN
4,2,etzoltim8ktgsaidhpebhnx79,5,2,4,1azk57wev30h044w949j46kpu,6jotswzwmrnex0mktyd575ez8,home,1,DL,Maddi Torre,NaN,NaN,NaN,NaN,NaN,NaN


#### Datos Agregados por Equipo

In [20]:
df_teamstats=swf.get_teamstats(df_partidos[df_partidos.match_id==df_partidos.match_id.values[-1]],ruta_dest)
print("Nº filas: {}".format(df_teamstats.shape[0]))
df_teamstats.head()

Nº filas: 2


,field,Aerial duels won,Blocked shots,Clearances,Corners won,Crosses,Crossing accuracy,Duels success rate,Fouls conceded,Goals,...,Red cards,Shooting accuracy,Shots,Shots inside the box,Shots on target,Shots outside the box,Tackles,Tackles success rate,Yellow cards,matchId
0,away,0.647,2,20,4,17,0.176,0.568,14,0,...,0,0.286,14,9,4,5,23,0.783,2,6jotswzwmrnex0mktyd575ez8
1,home,0.353,1,24,3,12,0.167,0.432,9,0,...,0,0.200,5,2,1,3,13,0.923,0,6jotswzwmrnex0mktyd575ez8
